# 0.0 Baseline

In [42]:
import pandas as pd


In [43]:
df_raw = pd.read_csv("/home/alvaro/Documentos/alvaro/comunidadeds/projetos/metodologiacrisp/dataset/HR-Employee-Attrition-balanced - HR-Employee-Attrition-balanced.csv")

In [45]:
# Research - Pesquisa de Mercado (SHRM)
# Entry-level (Cargos Operacionais): 40% do salário anual
# Professional (Especialistas): 100% do salário anual
# Manager (Gestores e Executivos): 200% do salário anual

baseline_df = df_raw.copy()

baseline_df['Annual_Salary'] = baseline_df['MonthlyIncome'] * 12
baseline_df['Replacement_Multiplier'] = baseline_df['JobLevel'].apply( lambda x: 0.40 if x <= 2 else 1.00 if x == 3 else 2.00 )
baseline_df['Replacement_Cost'] = baseline_df['Annual_Salary'] * baseline_df['Replacement_Multiplier']

# Isolando apenas as perdas (pessoas que pediram demissão)
leaving_df = baseline_df[baseline_df['Attrition'] == 'Yes']

# Metrics: 
total_employees = len(baseline_df)
employees_leaving = len(leaving_df)
attrition_rate = (employees_leaving / total_employees) * 100
avg_replacement_cost = leaving_df['Replacement_Cost'].mean()
total_annual_cost = leaving_df['Replacement_Cost'].sum()

print("="*60)
print("SITUAÇÃO ATUAL (BASELINE SEM MACHINE LEARNING)")
print("="*60)
print(f"Total de Funcionários: {total_employees}")
print(f"Funcionários que saem/ano (Rotatividade): {employees_leaving}")
print(f"Taxa de Rotatividade Anual: {attrition_rate:.2f}%")

print("-"*60)
print(f"Custo médio de substituição/pessoa: ${avg_replacement_cost:,.2f}")
print(f"Custo total anual de rotatividade: ${total_annual_cost:,.2f}")
print("="*60)


SITUAÇÃO ATUAL (BASELINE SEM MACHINE LEARNING)
Total de Funcionários: 1761
Funcionários que saem/ano (Rotatividade): 528
Taxa de Rotatividade Anual: 29.98%
------------------------------------------------------------
Custo médio de substituição/pessoa: $46,195.30
Custo total anual de rotatividade: $24,391,116.00


In [47]:
def simular_cenario_roi(recall, precision, taxa_retencao):

    # Constantes do Report
    CUSTO_MANUTENCAO_ANUAL = 20000
    CUSTO_INTERVENCAO = 5000

    # 1. Pessoas que a Máquina Aponta (Recall)
    verdadeiros_positivos = int(employees_leaving * recall)

    # 2. Pessoas Retidas de Fato pelo RH
    funcionarios_retidos = int(verdadeiros_positivos * taxa_retencao)

    # 3. Benefício Bruto (Pessoas salvas x O quanto a empresa gastaria pra demiti-las)
    economia_bruta = funcionarios_retidos * avg_replacement_cost
    
    # 5. Custos do Processo (Precisamos pagar pela manutenção de TI e pelas reuniões do RH)
    # Quantos alarmes o modelo gerou no total para conseguirmos os Verdadeiros Positivos?
    # Se a precisão é 70%, o volume total alardado é Verdadeiros Positivos / 0.70
    total_intencoes_rh = verdadeiros_positivos / precision if precision > 0 else 0
    custo_intervencoes = total_intencoes_rh * CUSTO_INTERVENCAO
    
    custos_anuais_totais = CUSTO_MANUTENCAO_ANUAL + custo_intervencoes
    
    # 6. Balanço Final
    economia_liquida = economia_bruta - custos_anuais_totais
    roi_anual = (economia_liquida / custos_anuais_totais) * 100 if custos_anuais_totais > 0 else 0
    
    return {
        "Cenário": "Simulado",
        "Precision": f"{precision*100:.0f}%",
        "Recall": f"{recall*100:.0f}%",
        "Retenção": f"{taxa_retencao*100:.0f}%",
        "Func. Retidos": funcionarios_retidos,
        "Economia Líquida (Anual)": f"${economia_liquida:,.0f}",
        "ROI Anual (%)": f"{roi_anual:.0f}%"
    }

# Criando tabela interativa com cenários conservador, moderado e otimista
tabela_analise = [
    simular_cenario_roi(precision = 0.60, recall=0.50, taxa_retencao=0.20),
    simular_cenario_roi(precision = 0.70, recall=0.60, taxa_retencao=0.30),
    simular_cenario_roi(precision = 0.80, recall=0.70, taxa_retencao=0.40),
]

df_potencial_roi = pd.DataFrame(tabela_analise)

# Renomear índices para identificar os cenários
df_potencial_roi['Cenário'] = ['Conservador', 'Moderado', 'Otimista']

display(df_potencial_roi)

,Cenário,Precision,Recall,Retenção,Func. Retidos,Economia Líquida (Anual),ROI Anual (%)
0,Conservador,60%,50%,20%,52,"$182,155",8%
1,Moderado,70%,60%,30%,94,"$2,065,215",91%
2,Otimista,80%,70%,40%,147,"$4,464,458",192%


In [ ]:
baseline_df.head().T